# Exemple de mosaïquage automatique

In [39]:
import geopandas as gpd
import subprocess
import pandas as pd
from geopandas import GeoDataFrame
from shapely import box
from shapely.ops import split

In [46]:
AOI1_SHP: str = "../data/fr/img/aoi1.shp"
AOI2_SHP: str = "../data/fr/img/aoi2.shp"
CUT_SHP: str = "../data/fr/merged/route_rail.shp"
CLIPPED_CUT_SHP: str = "../data/fr/clip/route_rail_clipped.shp"
PATH_SHP: str = "../data/fr/mosaic/cut_path.shp"
OUT_SHP: str = "../data/fr/out/final.shp"

MIN_X: int = 0
MIN_Y: int = 1
MAX_X: int = 2
MAX_Y: int = 3

In [48]:
aoi1_gdf: GeoDataFrame = gpd.read_file(AOI1_SHP)
aoi2_gdf: GeoDataFrame = gpd.read_file(AOI2_SHP)

aoi2_gdf = aoi2_gdf.to_crs(aoi1_gdf.crs)

area_gdf = pd.concat([aoi1_gdf, aoi2_gdf], ignore_index=True)
area_gdf = area_gdf.dissolve()
aoi_intersection = aoi1_gdf.intersection(aoi2_gdf.union_all())

### Identification et ajustement de l'overlap

In [10]:
width = aoi_intersection.total_bounds[MAX_X] - aoi_intersection.total_bounds[MIN_X]
height = aoi_intersection.total_bounds[MAX_Y] - aoi_intersection.total_bounds[MIN_Y]

if width >= height:
    print("Horizontal Mosaic")
    overlap_minx = min(aoi1_gdf.total_bounds[MIN_X], aoi2_gdf.total_bounds[MIN_X]) 
    overlap_maxx = max(aoi1_gdf.total_bounds[MAX_X], aoi2_gdf.total_bounds[MAX_X])
    overall_miny = max(aoi1_gdf.total_bounds[MIN_Y], aoi2_gdf.total_bounds[MIN_Y])
    overall_maxy = min(aoi1_gdf.total_bounds[MAX_Y], aoi2_gdf.total_bounds[MAX_Y])

    extended_overlap = box(overlap_minx, overall_miny, overlap_maxx, overall_maxy)
    extended_gdf: GeoDataFrame = GeoDataFrame({'geometry': [extended_overlap]}, crs=aoi1_gdf.crs)
else:
    print("Vertical Mosaic")
    overlap_minx = max(aoi1_gdf.total_bounds[MIN_X], aoi2_gdf.total_bounds[MIN_X]) 
    overlap_maxx = min(aoi1_gdf.total_bounds[MAX_X], aoi2_gdf.total_bounds[MAX_X])
    overall_miny = min(aoi1_gdf.total_bounds[MIN_Y], aoi2_gdf.total_bounds[MIN_Y])
    overall_maxy = max(aoi1_gdf.total_bounds[MAX_Y], aoi2_gdf.total_bounds[MAX_Y])

    extended_overlap = box(overlap_minx, overall_miny, overlap_maxx, overall_maxy)
    extended_gdf: GeoDataFrame = GeoDataFrame({'geometry': [extended_overlap]}, crs=aoi1_gdf.crs)

Vertical Mosaic


### Clipping du shp de coupe aux dimensions de l'overlap

In [20]:
cut_shp: GeoDataFrame = gpd.read_file(CUT_SHP).to_crs(aoi1_gdf.crs)
clipped_cut_shp: GeoDataFrame = gpd.clip(cut_shp, extended_gdf)
clipped_cut_shp.to_file(CLIPPED_CUT_SHP)

### Détermination du chemin de coupe et sauvegarde en shp

In [34]:
cut_box = clipped_cut_shp.total_bounds
offset = 0.01

if width < height:
    start_point = ((cut_box[MAX_X] + cut_box[MIN_X]) / 2, cut_box[MIN_Y] - offset)
    end_point = ((cut_box[MAX_X] + cut_box[MIN_X]) / 2, cut_box[MAX_Y] + offset)
else:
    start_point = ((cut_box[MAX_Y] + cut_box[MIN_Y]) / 2, cut_box[MIN_X] - offset)
    end_point = ((cut_box[MAX_Y] + cut_box[MIN_Y]) / 2, cut_box[MAX_X] + offset)

start_point_str: str = f"{float(start_point[0])},{float(start_point[1])}"
end_point_str: str = f"{float(end_point[0])},{float(end_point[1])}"

path_find = subprocess.run(f"""
docker run --rm -e QT_QPA_PLATFORM=offscreen -v ../data:/data \\
qgis/qgis /usr/bin/qgis_process run native:shortestpathpointtopoint \\
    --verbose \\
    --no-python \\
    --skip-loading-plugins \\
    --INPUT="/data/fr/clip/route_rail_clipped.shp" \\
    --STRATEGY=1 \\
    --START_POINT="{start_point_str}" \\
    --END_POINT="{end_point_str}" \\
    --OUTPUT="/data/fr/mosaic/cut_path.shp" \\
    --PROJECT_PATH="/data/qgis_project.qgz" \\
""", shell=True, capture_output=True, text=True)
print(path_find.stdout)
print(path_find.stderr)


----------------
Inputs
----------------

END_POINT:	4.206266898440441,48.17769422187318
INPUT:	/data/fr/clip/route_rail_clipped.shp
OUTPUT:	/data/fr/mosaic/cut_path.shp
START_POINT:	4.206266898440441,47.490281309285784
STRATEGY:	1


Building graph…
0...10...20...30...40...50...60...70...80...90...100 - done.
Calculating shortest path…
Writing results…

----------------
Results
----------------

OUTPUT:	/data/fr/mosaic/cut_path.shp
TRAVEL_COST:	1.8262958618299319

Detected locale "C" with character encoding "ANSI_X3.4-1968", which is not UTF-8.
Qt depends on a UTF-8 locale, and has switched to "C.UTF-8" instead.
If this causes problems, reconfigure your locale. See the locale(1) manual
for more information.
./src/core/providers/qgsproviderregistry.cpp:380 : (init) s] Loaded 36 providers (OAPIF;WFS;arcgisfeatureserver;arcgisimageserver;arcgismapserver;arcgisvectortileservice;cesiumtiles;copc;delimitedtext;ept;esrii3s;gdal;gpx;grass;grassraster;hana;mbtilesvectortiles;mdal;memory;mesh_m

### Coupe du shp de la zone avec le chemin précédemment obtenu

In [49]:
splitted_geometry = area_gdf.geometry.iloc[0]
cut_line = gpd.read_file(PATH_SHP)

split_result = split(splitted_geometry, cut_line.geometry.iloc[0])
shp_parts = [geom for geom in split_result.geoms if geom.geom_type == 'Polygon']

final_gdf = GeoDataFrame(geometry=shp_parts, crs=aoi1_gdf.crs)
final_gdf.to_file(OUT_SHP)

### Coupe des images et mosaïquage

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.merge import merge
from shapely.ops import split
from shapely.geometry import Polygon
import numpy as np

# Load your split line
split_line_gdf = gpd.read_file(PATH_SHP)
split_line = split_line_gdf.geometry.iloc[0]

# Load your two images
image1_path = "../data/fr/img/aoi1.tif"
image2_path = "../data/fr/img/aoi2.tif"

# Function to clip raster with a line (creates two halves)
def split_raster_with_line(raster_path, line_geom, output_prefix):
    with rasterio.open(raster_path) as src:
        # Get the raster bounds as a polygon
        bounds = src.bounds
        raster_polygon = Polygon([
            (bounds.left, bounds.bottom),
            (bounds.right, bounds.bottom),
            (bounds.right, bounds.top),
            (bounds.left, bounds.top)
        ])
        
        # Split the raster polygon with the line
        from shapely.ops import split
        split_polygons = split(raster_polygon, line_geom)
        
        # Clip and save each half
        clipped_images = []
        for i, poly in enumerate(split_polygons.geoms):
            # Convert to GeoJSON-like format
            geoms = [poly.__geo_interface__]
            out_image, out_transform = mask(src, geoms, crop=True)
            
            # Save the clipped half
            output_path = f"../data/fr/out/tmp{output_prefix}_part_{i+1}.tiff"
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            
            with rasterio.open(output_path, "w", **out_meta) as dest:
                dest.write(out_image)
            
            clipped_images.append(output_path)
        
        return clipped_images

# Split both images
image1_parts = split_raster_with_line(image1_path, split_line, "image1")
image2_parts = split_raster_with_line(image2_path, split_line, "image2")

# Now merge the corresponding halves
# Option A: Merge all parts together
all_parts = image1_parts + image2_parts

# Open all raster files
src_files = [rasterio.open(f) for f in all_parts]

# Merge them
merged_image, merged_transform = merge(src_files)

# Save the final merged image
out_meta = src_files[0].meta.copy()
out_meta.update({
    "height": merged_image.shape[1],
    "width": merged_image.shape[2],
    "transform": merged_transform
})

with rasterio.open("../data/fr/img/final_merged_image.tiff", "w", **out_meta) as dest:
    dest.write(merged_image)

# Close all files
for src in src_files:
    src.close()